# Evaluation
This notebook evaluates all three trained models using accuracy, classification reports, and visualizations.

In [ ]:
# matplotlib and seaborn are used to create and style the evaluation plots
# sklearn.metrics provides accuracy_score, confusion_matrix, and classification_report
# os.makedirs ensures the figures directory exists before saving any plots
import os
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

FIGURES_DIR = '../figures'
os.makedirs(FIGURES_DIR, exist_ok=True)

### Setup — Train Models
*(Runs the pipeline steps so this notebook works on its own.)*

In [ ]:
# reproducing the full pipeline so this notebook can run independently
# loading the data, dropping non-informative columns, encoding categoricals
df = pd.read_csv('../data/alzheimers_disease_data.csv')
df = df.drop(columns=[c for c in ['PatientID', 'DoctorInCharge'] if c in df.columns])

y = df['Diagnosis'].astype(int)
X = pd.get_dummies(df.drop(columns=['Diagnosis']), drop_first=True)

# stratified 80/20 split to maintain the original class ratio in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# scaler is fitted on training data only to avoid data leakage into the test set
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Logistic Regression requires standardized features; tree-based models do not
SCALED_MODELS = {'Logistic Regression'}
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':             XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1,
                                         eval_metric='logloss', random_state=42, n_jobs=-1),
}

# training each model and immediately generating test set predictions
fitted = {}
predictions = {}

for name, model in models.items():
    if name in SCALED_MODELS:
        model.fit(X_train_scaled, y_train)
        predictions[name] = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        predictions[name] = model.predict(X_test)
    fitted[name] = model

print('All models trained.')

### Accuracy

In [ ]:
# accuracy_score computes the fraction of correctly classified samples
# calculated for all three models and stored in a dictionary for easy comparison
accuracies = {name: accuracy_score(y_test, y_pred) for name, y_pred in predictions.items()}

for name, acc in accuracies.items():
    print(f'{name}: {acc:.4f}')

### Classification Report

In [ ]:
# accuracy alone can be misleading, especially with imbalanced datasets
# precision: of all positive predictions, what fraction were truly positive
# recall: of all actual positive cases, what fraction did the model correctly identify
# f1-score: harmonic mean of precision and recall — balances both metrics in a single value
# support: the number of actual samples in each class within the test set
for name, y_pred in predictions.items():
    print(f'\n--- {name} ---')
    print(classification_report(y_test, y_pred, target_names=['No', 'Yes']))

### Confusion Matrices

In [ ]:
# a confusion matrix breaks down predictions into four categories:
# true positives (TP): correctly predicted as Alzheimer's
# true negatives (TN): correctly predicted as healthy
# false positives (FP): predicted Alzheimer's but actually healthy
# false negatives (FN): predicted healthy but actually Alzheimer's — the most critical error in medical diagnosis
# all three matrices are plotted side by side for a direct visual comparison
fig, axes = plt.subplots(1, len(predictions), figsize=(5 * len(predictions), 4))

for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=['No', 'Yes'], yticklabels=['No', 'Yes'], ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/confusion_matrices.png', dpi=120)
plt.show()

### Feature Importance

In [ ]:
# feature importance measures how much each input feature contributed to the model's predictions
# Random Forest calculates this by tracking how much each feature reduces impurity across all trees
# features are sorted ascending so the most important one appears at the top of the chart
# this is useful for understanding which clinical factors are most associated with the diagnosis
importances = (
    pd.Series(fitted['Random Forest'].feature_importances_, index=X.columns)
    .sort_values(ascending=True)
    .tail(15)
)

plt.figure(figsize=(8, 6))
importances.plot(kind='barh', color='steelblue')
plt.title('Top 15 Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/feature_importance.png', dpi=120)
plt.show()

### Accuracy Comparison

In [ ]:
# bar chart comparing the test accuracy of all three models
# the y-axis is zoomed in by starting just below the lowest score
# this makes differences between models visible that would otherwise appear flat at the top
names  = list(accuracies.keys())
scores = list(accuracies.values())

plt.figure(figsize=(7, 4))
bars = plt.bar(names, scores, color=['#4C72B0', '#55A868', '#C44E52'])

plt.ylim(max(0.0, min(scores) - 0.05), 1.0)
plt.ylabel('Accuracy')
plt.title('Model Accuracy Comparison')

# annotating each bar with its exact accuracy value for precise reading
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width() / 2, score + 0.005,
             f'{score:.4f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/accuracy_comparison.png', dpi=120)
plt.show()

### Summary

In [ ]:
# a majority class baseline predicts the most frequent label for every sample
# it represents the minimum bar a model must clear to be considered useful
# comparing each model's accuracy against the baseline shows the actual gain from learning
baseline   = y_test.value_counts(normalize=True).max()
best_model = max(accuracies, key=accuracies.get)

print('=== Summary ===')
print(f'Baseline (majority class): {baseline:.4f}')
for name, acc in accuracies.items():
    print(f'{name:22s}  accuracy={acc:.4f}  (+{acc - baseline:.4f} vs baseline)')

print(f'\nBest model: {best_model} ({accuracies[best_model]:.4f})')

# top 5 features by importance — reversed so the highest importance appears first
print('\nTop 5 features (Random Forest):')
print(importances.tail(5)[::-1].round(4).to_string())